# 📝 Arabic Poetry Generation using Fine-Tuned AraGPT2

This project focuses on generating Arabic poetry by fine-tuning the AraGPT2 model on a rich dataset of Arabic poems.
We explore how tuning parameters like sample size, number of epochs, and learning rate impact the quality of generated poems.

**Evaluation Metrics:**
- Loss
- Perplexity
- Human Scoring

**Model:** [AraGPT2-base](https://huggingface.co/aubmindlab/aragpt2-base)
**Dataset:** [ashaar on Hugging Face](https://huggingface.co/datasets/arbml/ashaar)


## 1. Environment Setup

In [1]:
!pip install datasets pytorch-lightning accelerate
!pip install transformers==4.17
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.1/823.1 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Load the Dataset

We use the `arbml/ashaar` dataset from Hugging Face, which contains Arabic poems with metadata such as:
- Poem Title
- verses
- Meter
- Theme
- Poet information

In [2]:
from datasets import load_dataset

# Load the dataset directly from Hugging Face
dataset = load_dataset("arbml/ashaar")

# Check the structure
print(dataset)
print(dataset["train"][0])


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.71k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/34.7k [00:00<?, ?B/s]

train-00000-of-00002.parquet:   0%|          | 0.00/126M [00:00<?, ?B/s]

train-00001-of-00002.parquet:   0%|          | 0.00/151M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/254630 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['poem title', 'poem meter', 'poem verses', 'poem theme', 'poem url', 'poet name', 'poet description', 'poet url', 'poet era', 'poet location', 'poem description', 'poem language type'],
        num_rows: 254630
    })
})
{'poem title': 'أصبح الملك للذي فطر الخلق', 'poem meter': 'بحر الخفيف', 'poem verses': ['أَصبَحَ المُلك لِلَّذي فَطر الخَل', 'قَ بِتَقديرٍ للعَزيز العَليمِ', 'غافر الذَنب للمسيءِ بِعَفوٍ', 'قابل التَوب ذي العَطاء العَميمِ', 'مُرسل المُصطَفى البَشير إِلَينا', 'رَحمة مِنهُ بِالكَلام القَديمِ', 'رَبَنا رَبّنا إِلَيكَ أَنينا', 'فَأَجرنا مِن حَر نار الجَحيمِ', 'وَاكفِنا شَرّ ما نَخاف بِلُطفٍ', 'يا عَظيماً يَرجى لِكُل عَظيمِ', 'وَتَقبل أَعمالَنا وَاعفُ عَنا', 'وَأَنلنا دُخول دار النَعيمِ', 'بِنَبي بَعثَتهُ فَهَدانا', 'لِصِراط مِن الهُدى مُستَقيمِ', 'وَبِمَن نَحنُ في حِماهُ مَدى الدَهر', 'أَخيهِ يَحيى الحصور الكَريمِ', 'أَدرك أَدرك قَوماً أَتوا بافتقار', 'وَاِنكِسار وَمَدمَع مَسجومِ', 'شَهدت أَرواحَهُم أَنكَ اللَهُ', 'وَجا

## 3. Dataset Preprocessing

We:
- Remove diacritics
- Normalize spaces and punctuation
- Add a special prompt (e.g., `[Love]`) at the start of each example based on the poem theme.



In [3]:
import re

def clean_verses(verses):
    # verses is a list → join into a single string
    if isinstance(verses, list):
        verses = '، '.join(verses)  # Join with Arabic comma or space

    # Remove diacritics
    verses = re.sub(r'[\u064B-\u0652]', '', verses)
    # Remove extra spaces and unwanted symbols
    verses = re.sub(r'\s+', ' ', verses)
    # Ensure each verse is on a new line (split on Arabic comma or period)
    verses = verses.replace('،', '\n')
    verses = verses.replace('.', '\n')
    return verses.strip()

def add_prompt(example):
    cleaned_verses = clean_verses(example['poem verses'])
    return {'input_text': f"[{example['poem theme']}] {cleaned_verses}"}



# Apply the map function with the correct fields
dataset = dataset.map(add_prompt)

Map:   0%|          | 0/254630 [00:00<?, ? examples/s]

In [4]:
print(dataset["train"][0]["input_text"])

[قصيدة دينية] أصبح الملك للذي فطر الخل
 ق بتقدير للعزيز العليم
 غافر الذنب للمسيء بعفو
 قابل التوب ذي العطاء العميم
 مرسل المصطفى البشير إلينا
 رحمة منه بالكلام القديم
 ربنا ربنا إليك أنينا
 فأجرنا من حر نار الجحيم
 واكفنا شر ما نخاف بلطف
 يا عظيما يرجى لكل عظيم
 وتقبل أعمالنا واعف عنا
 وأنلنا دخول دار النعيم
 بنبي بعثته فهدانا
 لصراط من الهدى مستقيم
 وبمن نحن في حماه مدى الدهر
 أخيه يحيى الحصور الكريم
 أدرك أدرك قوما أتوا بافتقار
 وانكسار ومدمع مسجوم
 شهدت أرواحهم أنك الله
 وجاءوا بكل قلب سليم


## 4. Tokenization

We tokenize the input using AraGPT2’s tokenizer, ensuring:
- Truncation to 256 tokens
- Padding
- Consistent special tokens


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("aubmindlab/aragpt2-base", use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(example):
    return tokenizer(example["input_text"], truncation=True, padding="max_length", max_length=256)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# this important line to remove unnecessary columns:
tokenized_datasets = tokenized_datasets.remove_columns(
    ["poem title", "poem meter", "poem verses", "poem theme", "poem url",
     "poet name", "poet description", "poet url", "poet era", "poet location",
     "poem description", "poem language type", "input_text"]
)

print(tokenized_datasets)


Downloading:   0%|          | 0.00/843 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.85M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.43M [00:00<?, ?B/s]

Map:   0%|          | 0/254630 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 254630
    })
})


In [6]:
print(tokenized_datasets)


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 254630
    })
})


# 🔴Experiment 1: Small Sample, Low Epochs (Baseline)

**Training Configuration:**
- Sample size: **5,000**
- Epochs: **2**
- Learning rate: **5e-5**


## 1.1. Training Setup


## 1.1.1. Load Pretrained AraGPT2

We load the `aubmindlab/aragpt2-base` model and fine-tune it on our poetry data.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("aubmindlab/aragpt2-base")


Downloading:   0%|          | 0.00/527M [00:00<?, ?B/s]

## 1.1.2. Dataset Splitting and Sampling

We split the dataset into training (90%) and evaluation (10%).  .

In [ ]:
# Split the dataset normally (to respect random selection)
split_dataset = tokenized_datasets["train"].train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# Select a smaller subset for faster training
small_train_dataset = train_dataset.select(range(5000))  # 5,000 examples for training
small_eval_dataset = eval_dataset.select(range(500))    # 500 examples for evaluation

print(f"Small train examples: {len(small_train_dataset)}")
print(f"Small eval examples: {len(small_eval_dataset)}")

Small train examples: 5000
Small eval examples: 500


## 1. 2. Define Training Configuration

We specify:
- Learning rate
- Epochs
- Batch size
- Logging and saving strategy


In [ ]:
from transformers import TrainingArguments, get_scheduler

training_args = TrainingArguments(
    output_dir="./aragpt2-ashaar-finetuned-gpu",
    evaluation_strategy="epoch",
    learning_rate=5e-5,                     # 5e-5 for learning rate
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=2,                     # 2 for epochs
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs",
    report_to="none",
    fp16=True,                              # Mixed precision (faster on GPU)
    gradient_accumulation_steps=2 ,
    warmup_steps=500,                       # Gradually increase learning rate
    lr_scheduler_type="linear"
)

## 1. 3. Initialize Trainer

Using Hugging Face’s `Trainer` class to manage training and evaluation.


In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


Using amp half precision backend
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:474: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


## 1. 4. Training the Model

In [ ]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 5000
  Num Epochs = 2
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 2
  Total optimization steps = 624
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:1949: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx_manager = autocast(dtype=self.amp_dtype)


Epoch,Training Loss,Validation Loss
0,No log,4.816352
1,6.544300,4.512102


***** Running Evaluation *****
  Num examples = 500
  Batch size = 4
Saving model checkpoint to ./aragpt2-ashaar-finetuned-gpu/checkpoint-312
Configuration saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-312/config.json
Model weights saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-312/pytorch_model.bin
tokenizer config file saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-312/tokenizer_config.json
Special tokens file saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-312/special_tokens_map.json
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:1949: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx_manager = autocast(dtype=self.amp_dtype)
***** Running Evaluation *****
  Num examples = 500
  Batch size = 4
Saving model checkpoint to ./aragpt2-ashaar-finetuned-gpu/checkpoint-624
Configuration saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-624/config.json
Model weights saved in ./arag

TrainOutput(global_step=624, training_loss=6.202695993276743, metrics={'train_runtime': 354.7608, 'train_samples_per_second': 28.188, 'train_steps_per_second': 1.759, 'total_flos': 1305414991872000.0, 'train_loss': 6.202695993276743, 'epoch': 2.0})

## 1. 5. Save the Fine-Tuned Model

We save the model and tokenizer for each experiment under distinct output directories (e.g., `experiment1-aragpt2-finetuned`).


In [ ]:
trainer.save_model("./experiment1-aragpt2-finetuned")           # Saves the model
tokenizer.save_pretrained("./experiment1-aragpt2-finetuned")    # Saves the tokenizer


Saving model checkpoint to ./experiment1-aragpt2-finetuned
Configuration saved in ./experiment1-aragpt2-finetuned/config.json
Model weights saved in ./experiment1-aragpt2-finetuned/pytorch_model.bin
tokenizer config file saved in ./experiment1-aragpt2-finetuned/tokenizer_config.json
Special tokens file saved in ./experiment1-aragpt2-finetuned/special_tokens_map.json
tokenizer config file saved in ./experiment1-aragpt2-finetuned/tokenizer_config.json
Special tokens file saved in ./experiment1-aragpt2-finetuned/special_tokens_map.json


('./experiment1-aragpt2-finetuned/tokenizer_config.json',
 './experiment1-aragpt2-finetuned/special_tokens_map.json',
 './experiment1-aragpt2-finetuned/vocab.json',
 './experiment1-aragpt2-finetuned/merges.txt',
 './experiment1-aragpt2-finetuned/added_tokens.json')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./experiment1-aragpt2-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(model_path)

# Move model to GPU manually if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


Didn't find file ./experiment1-aragpt2-finetuned/added_tokens.json. We won't load it.
loading file ./experiment1-aragpt2-finetuned/vocab.json
loading file ./experiment1-aragpt2-finetuned/merges.txt
loading file None
loading file ./experiment1-aragpt2-finetuned/special_tokens_map.json
loading file ./experiment1-aragpt2-finetuned/tokenizer_config.json
loading configuration file ./experiment1-aragpt2-finetuned/config.json
Model config GPT2Config {
  "_name_or_path": "./experiment1-aragpt2-finetuned",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 0,
  "embd_pdrop": 0.1,
  "eos_token_id": 0,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false

## 1. 6. Poetry Generation

We generate multiple Arabic poems given a theme-based prompt like `[قصيدة حزينة]`, using top-k and top-p sampling.


In [ ]:
def generate_multiple_poems(prompt, max_length=70, top_k=40, top_p=0.85, temperature=1.0, num_return_sequences=3):
    device = model.device
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    outputs = model.generate(
        input_ids,
        max_length=max_length,
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.8,
        no_repeat_ngram_size=4,
        early_stopping=True
    )

    poems = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return poems
prompt = "[قصيدة حزينة]"
poems = generate_multiple_poems(prompt, top_k=40, top_p=0.85, num_return_sequences=3)
for idx, poem in enumerate(poems, 1):
    print(f"Poem {idx}:\n{poem}\n")


Poem 1:
[قصيدة حزينة] إن كنت لا ترى
 أو لي ولا تعقليني إلى غيرك 
 فلا أرى فيك من خير سوى أن يكون في فضل العذل والعزم إلا وقد نلت كل حسن ما نالوا به الدهر منه غير مماتك ولم يعد منهم فما أبغضا
 وما عليكما ذا لم تبغي يوما دون هذا الفضل فذا لك

Poem 2:
[قصيدة حزينة] هل إذا حماك من أدمعك
 ماذا في دمعم القلب أمسى على دموعكم قد بكى علي 
 ؟!؟ وما الذي أسأمه البكاء مني حين يبكي ولا الدمع وقد أهفو عني أن أرخنت عن جفني ريقها: ودمي يا ليت دموعي إلى عينيك كيف تبكيه

Poem 3:
[قصيدة حزينة] لا يـيل كي هلوه
 ممات فُتيجن نمول بلوا حيههذا عللا 
 وأغمارا شيامهم قد ضلوا علم تيضاهمته فضياه جيادنا في ناره رقمه وقد غياطة ملهم فيه الأ



### ✅ *Human Evaluation – Experiment 1*

* *Poem 1:* Strange mix of philosophical ideas and disjointed metaphors. Moderate clarity. Some nonsense.
* *Poem 2:* Emotional but structurally weak. Slight repetition.
* *Poem 3:* Contains gibberish at the end.

*Average Scores:*

* Fluency: 3
* Coherence: 2.5
* Structure: 2
* Emotion: 3

*Human Score:* 2.5 / 5

## 1. 7. Quantitative Evaluation

We compute:
- **Loss** on the evaluation set
- **Perplexity** as `exp(loss)` to assess how confidently the model predicts the next token


In [ ]:
import math
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import default_data_collator

# Make sure model is in evaluation mode
model.eval()

# Create a DataLoader for evaluation
eval_loader = DataLoader(
    small_eval_dataset,
    batch_size=4,
    collate_fn=default_data_collator
)

# Track total loss
total_loss = 0
n_batches = 0

# Disable gradient calculation for evaluation
with torch.no_grad():
    for batch in tqdm(eval_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
        loss = outputs.loss

        total_loss += loss.item()
        n_batches += 1

Evaluating: 100%|██████████| 125/125 [00:12<00:00,  9.72it/s]


In [ ]:
# Calculate Average Loss
avg_loss = total_loss / n_batches

print(f"\n Evaluation Loss: {avg_loss:.4f}")



 Evaluation Loss: 7.7391


In [ ]:
# Calculate Perplexity
perplexity = math.exp(avg_loss)

print(f" Perplexity: {perplexity:.2f}")

 Perplexity: 2296.43


# 🔴Experiment 2: Medium Sample, Same Epochs

**Training Configuration:**
- Sample size: **10,000**
- Epochs: **2**
- Learning rate: **5e-5**


## 2.1. Training Setup


## 2.1.1. Load Pretrained AraGPT2

We load the `aubmindlab/aragpt2-base` model and fine-tune it on our poetry data.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("aubmindlab/aragpt2-base")


loading configuration file https://huggingface.co/aubmindlab/aragpt2-base/resolve/main/config.json from cache at /root/.cache/huggingface/transformers/c8489d06657d6f7bbc7c50a9ebb22c5a0d744f1e3abd4c1fb4552c1de84a7d16.c27f94d1fe6a71d31824df935e650dfdda7292d7e26004c12d272f15abbb677c
Model config GPT2Config {
  "_name_or_path": "aubmindlab/aragpt2-base",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 0,
  "embd_pdrop": 0.1,
  "eos_token_id": 0,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls

## 2.1.2. Dataset Splitting and Sampling

We split the dataset into training (90%) and evaluation (10%).  .

In [ ]:
# Split the dataset normally (to respect random selection)
split_dataset = tokenized_datasets["train"].train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# Select a smaller subset for faster training
small_train_dataset = train_dataset.select(range(10000))  # 10,000 examples for training
small_eval_dataset = eval_dataset.select(range(1000))    # 1000 examples for evaluation

print(f"Small train examples: {len(small_train_dataset)}")
print(f"Small eval examples: {len(small_eval_dataset)}")

Small train examples: 10000
Small eval examples: 1000


## 2. 2. Define Training Configuration

We specify:
- Learning rate
- Epochs
- Batch size
- Logging and saving strategy


In [ ]:
from transformers import TrainingArguments, get_scheduler

training_args = TrainingArguments(
    output_dir="./aragpt2-ashaar-finetuned-gpu",
    evaluation_strategy="epoch",
    learning_rate=5e-5,                           # 5e-5 for learning rate
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=2,                           # 2 for epochs
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs",
    report_to="none",
    fp16=True,                                    # Mixed precision (faster on GPU)
    gradient_accumulation_steps=2 ,
    warmup_steps=500,                             # Gradually increase learning rate
    lr_scheduler_type="linear"
)

PyTorch: setting up devices


## 2. 3. Initialize Trainer

Using Hugging Face’s `Trainer` class to manage training and evaluation.


In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


Using amp half precision backend


## 2. 4. Training the Model

In [ ]:
trainer.train()

***** Running training *****
  Num examples = 10000
  Num Epochs = 2
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 2
  Total optimization steps = 1250


Epoch,Training Loss,Validation Loss
1,6.420500,4.475285
2,4.702800,4.286901


***** Running Evaluation *****
  Num examples = 1000
  Batch size = 4
Saving model checkpoint to ./aragpt2-ashaar-finetuned-gpu/checkpoint-625
Configuration saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-625/config.json
Model weights saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-625/pytorch_model.bin
tokenizer config file saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-625/tokenizer_config.json
Special tokens file saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-625/special_tokens_map.json
Deleting older checkpoint [aragpt2-ashaar-finetuned-gpu/checkpoint-312] due to args.save_total_limit
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:1949: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx_manager = autocast(dtype=self.amp_dtype)
***** Running Evaluation *****
  Num examples = 1000
  Batch size = 4
Saving model checkpoint to ./aragpt2-ashaar-finetuned-gpu/checkpoint-1250
Config

TrainOutput(global_step=1250, training_loss=5.38210322265625, metrics={'train_runtime': 691.3411, 'train_samples_per_second': 28.929, 'train_steps_per_second': 1.808, 'total_flos': 2612920320000000.0, 'train_loss': 5.38210322265625, 'epoch': 2.0})

## 2. 5. Save the Fine-Tuned Model

We save the model and tokenizer for each experiment under distinct output directories (e.g., `experiment1-aragpt2-finetuned`).


In [ ]:
trainer.save_model("./experiment2-aragpt2-finetuned")           # Saves the model
tokenizer.save_pretrained("./experiment2-aragpt2-finetuned")    # Saves the tokenizer


Saving model checkpoint to ./experiment2-aragpt2-finetuned
Configuration saved in ./experiment2-aragpt2-finetuned/config.json
Model weights saved in ./experiment2-aragpt2-finetuned/pytorch_model.bin
tokenizer config file saved in ./experiment2-aragpt2-finetuned/tokenizer_config.json
Special tokens file saved in ./experiment2-aragpt2-finetuned/special_tokens_map.json
tokenizer config file saved in ./experiment2-aragpt2-finetuned/tokenizer_config.json
Special tokens file saved in ./experiment2-aragpt2-finetuned/special_tokens_map.json


('./experiment2-aragpt2-finetuned/tokenizer_config.json',
 './experiment2-aragpt2-finetuned/special_tokens_map.json',
 './experiment2-aragpt2-finetuned/vocab.json',
 './experiment2-aragpt2-finetuned/merges.txt',
 './experiment2-aragpt2-finetuned/added_tokens.json')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./experiment2-aragpt2-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(model_path)

# Move model to GPU manually if available:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


Didn't find file ./experiment2-aragpt2-finetuned/added_tokens.json. We won't load it.
loading file ./experiment2-aragpt2-finetuned/vocab.json
loading file ./experiment2-aragpt2-finetuned/merges.txt
loading file None
loading file ./experiment2-aragpt2-finetuned/special_tokens_map.json
loading file ./experiment2-aragpt2-finetuned/tokenizer_config.json
loading configuration file ./experiment2-aragpt2-finetuned/config.json
Model config GPT2Config {
  "_name_or_path": "./experiment2-aragpt2-finetuned",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 0,
  "embd_pdrop": 0.1,
  "eos_token_id": 0,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false

## 2. 6. Poetry Generation

We generate multiple Arabic poems given a theme-based prompt like `[قصيدة حزينة]`, using top-k and top-p sampling.


In [ ]:
def generate_multiple_poems(prompt, max_length=70, top_k=40, top_p=0.85, temperature=1.0, num_return_sequences=3):
    device = model.device
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    outputs = model.generate(
        input_ids,
        max_length=max_length,
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.8,
        no_repeat_ngram_size=4,
        early_stopping=True
    )

    poems = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return poems
prompt = "[قصيدة حزينة]"
poems = generate_multiple_poems(prompt, top_k=40, top_p=0.85, num_return_sequences=3)
for idx, poem in enumerate(poems, 1):
    print(f"Poem {idx}:\n{poem}\n")


Poem 1:
[قصيدة حزينة] رأيت بعيني على الغيد
 فرأيت أنما قلت : كيف كنت أمسى يوما في الهوى من سقمها وجنونا 
 فقلت: يا ليتي التي أهواها لم أر مني قطاكا!
 قال- ماذا فعلت حين قالت؟ أين ذاك ؟ أنا أضحك الآن إذا ما مضى!"يا لكلي إن

Poem 2:
[قصيدة حزينة] لا أدري ما الذي قد رأى
 وما كنت أعرف أن لي من علم 
 فما كان يعلم أنه يدري شيئا ولا يدريه أحدا إذا سمعا له خبرا وليس يرى سواه فإن لم يكن يعرف إلا وهو ليس يدركها فإنه يسمع به يقينا فهو يخاف غدا إذ يبصره؟ فكيف يا أخي في الهوى ؟ كيف أنت تعلم أني رأيت

Poem 3:
[قصيدة حزينة] ألا ما قد مضى الدهر من عهد
 وما كان هذا الليل إلا به
 يا أيها الزمان لا ملامة ولا ندمت إن لم تكن ذا حال 
 في الدنيا معجلة أو سقامها� ومن الذي بات يوما أنكره على الناس فيه عزمة ردى
 ولو كنت له ضنورا لما أعيتني



### ✅ *Human Evaluation – Experiment 2*

* *Poem 1:* Surprisingly structured with vivid imagery. Good poetic intent.
* *Poem 2:* Makes sense logically but lacks emotional appeal.
* *Poem 3:* Dense but expressive. Slightly abstract yet meaningful.

*Average Scores:*

* Fluency: 3.5
* Coherence: 3
* Structure: 3.5
* Emotion: 2.8

*Human Score:* 3.2 / 5


## 2. 7. Quantitative Evaluation

We compute:
- **Loss** on the evaluation set
- **Perplexity** as `exp(loss)` to assess how confidently the model predicts the next token


In [ ]:
import math
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import default_data_collator

# Make sure model is in evaluation mode
model.eval()

# Create a DataLoader for evaluation
eval_loader = DataLoader(
    small_eval_dataset,
    batch_size=4,
    collate_fn=default_data_collator
)

# Track total loss
total_loss = 0
n_batches = 0

# Disable gradient calculation for evaluation
with torch.no_grad():
    for batch in tqdm(eval_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
        loss = outputs.loss

        total_loss += loss.item()
        n_batches += 1

Evaluating: 100%|██████████| 250/250 [00:26<00:00,  9.44it/s]


In [ ]:
# Calculate Average Loss
avg_loss = total_loss / n_batches

print(f"\n Evaluation Loss: {avg_loss:.4f}")



 Evaluation Loss: 8.5515


In [ ]:
# Calculate Perplexity
perplexity = math.exp(avg_loss)

print(f" Perplexity: {perplexity:.2f}")

 Perplexity: 5174.41


# 🔴Experiment 3: Full Sample, Same Epochs

**Training Configuration:**
- Sample size: **50,000**
- Epochs: **2**
- Learning rate: **5e-5**


## 3.1. Training Setup


## 3.1.1. Load Pretrained AraGPT2

We load the `aubmindlab/aragpt2-base` model and fine-tune it on our poetry data.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("aubmindlab/aragpt2-base")


loading configuration file https://huggingface.co/aubmindlab/aragpt2-base/resolve/main/config.json from cache at /root/.cache/huggingface/transformers/c8489d06657d6f7bbc7c50a9ebb22c5a0d744f1e3abd4c1fb4552c1de84a7d16.c27f94d1fe6a71d31824df935e650dfdda7292d7e26004c12d272f15abbb677c
Model config GPT2Config {
  "_name_or_path": "aubmindlab/aragpt2-base",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 0,
  "embd_pdrop": 0.1,
  "eos_token_id": 0,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls

## 3.1.2. Dataset Splitting and Sampling

We split the dataset into training (90%) and evaluation (10%).  .

In [ ]:
# Split the dataset normally (to respect random selection)
split_dataset = tokenized_datasets["train"].train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# Select a smaller subset for faster training
small_train_dataset = train_dataset.select(range(50000))  # 50,000 examples for training
small_eval_dataset = eval_dataset.select(range(5000))    # 5000 examples for evaluation

print(f"Small train examples: {len(small_train_dataset)}")
print(f"Small eval examples: {len(small_eval_dataset)}")

Small train examples: 50000
Small eval examples: 5000


## 3. 2. Define Training Configuration

We specify:
- Learning rate
- Epochs
- Batch size
- Logging and saving strategy


In [ ]:
from transformers import TrainingArguments, get_scheduler

training_args = TrainingArguments(
    output_dir="./aragpt2-ashaar-finetuned-gpu",
    evaluation_strategy="epoch",
    learning_rate=5e-5,                             # 5e-5 for learning rate
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=2,                             # 2 for epochs
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs",
    report_to="none",
    fp16=True,                                      # Mixed precision (faster on GPU)
    gradient_accumulation_steps=2 ,
    warmup_steps=500,                               # Gradually increase learning rate
    lr_scheduler_type="linear"
)

PyTorch: setting up devices


## 3. 3. Initialize Trainer

Using Hugging Face’s `Trainer` class to manage training and evaluation.


In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


Using amp half precision backend
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:474: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


## 3. 4. Training the Model

In [ ]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 50000
  Num Epochs = 2
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 2
  Total optimization steps = 6250
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:1949: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx_manager = autocast(dtype=self.amp_dtype)


Epoch,Training Loss,Validation Loss
1,4.341100,4.070418
2,4.222300,3.979146


***** Running Evaluation *****
  Num examples = 5000
  Batch size = 4
Saving model checkpoint to ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125
Configuration saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/config.json
Model weights saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/pytorch_model.bin
tokenizer config file saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/tokenizer_config.json
Special tokens file saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/special_tokens_map.json
Deleting older checkpoint [aragpt2-ashaar-finetuned-gpu/checkpoint-625] due to args.save_total_limit
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:1949: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx_manager = autocast(dtype=self.amp_dtype)
***** Running Evaluation *****
  Num examples = 5000
  Batch size = 4
Saving model checkpoint to ./aragpt2-ashaar-finetuned-gpu/checkpoint-6250
C

TrainOutput(global_step=6250, training_loss=4.53793626953125, metrics={'train_runtime': 3345.0355, 'train_samples_per_second': 29.895, 'train_steps_per_second': 1.868, 'total_flos': 1.30646016e+16, 'train_loss': 4.53793626953125, 'epoch': 2.0})

## 3. 5. Save the Fine-Tuned Model

We save the model and tokenizer for each experiment under distinct output directories (e.g., `experiment1-aragpt2-finetuned`).


In [ ]:
trainer.save_model("./experiment3-aragpt2-finetuned")           # Saves the model
tokenizer.save_pretrained("./experiment3-aragpt2-finetuned")    # Saves the tokenizer


Saving model checkpoint to ./experiment3-aragpt2-finetuned
Configuration saved in ./experiment3-aragpt2-finetuned/config.json
Model weights saved in ./experiment3-aragpt2-finetuned/pytorch_model.bin
tokenizer config file saved in ./experiment3-aragpt2-finetuned/tokenizer_config.json
Special tokens file saved in ./experiment3-aragpt2-finetuned/special_tokens_map.json
tokenizer config file saved in ./experiment3-aragpt2-finetuned/tokenizer_config.json
Special tokens file saved in ./experiment3-aragpt2-finetuned/special_tokens_map.json


('./experiment3-aragpt2-finetuned/tokenizer_config.json',
 './experiment3-aragpt2-finetuned/special_tokens_map.json',
 './experiment3-aragpt2-finetuned/vocab.json',
 './experiment3-aragpt2-finetuned/merges.txt',
 './experiment3-aragpt2-finetuned/added_tokens.json')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./experiment3-aragpt2-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(model_path)

# Move model to GPU manually if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


Didn't find file ./experiment3-aragpt2-finetuned/added_tokens.json. We won't load it.
loading file ./experiment3-aragpt2-finetuned/vocab.json
loading file ./experiment3-aragpt2-finetuned/merges.txt
loading file None
loading file ./experiment3-aragpt2-finetuned/special_tokens_map.json
loading file ./experiment3-aragpt2-finetuned/tokenizer_config.json
loading configuration file ./experiment3-aragpt2-finetuned/config.json
Model config GPT2Config {
  "_name_or_path": "./experiment3-aragpt2-finetuned",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 0,
  "embd_pdrop": 0.1,
  "eos_token_id": 0,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false

## 3. 6. Poetry Generation

We generate multiple Arabic poems given a theme-based prompt like `[قصيدة حزينة]`, using top-k and top-p sampling.


In [ ]:
def generate_multiple_poems(prompt, max_length=70, top_k=40, top_p=0.85, temperature=1.0, num_return_sequences=3):
    device = model.device
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    outputs = model.generate(
        input_ids,
        max_length=max_length,
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.8,
        no_repeat_ngram_size=4,
        early_stopping=True
    )

    poems = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return poems
prompt = "[قصيدة حزينة]"
poems = generate_multiple_poems(prompt, top_k=40, top_p=0.85, num_return_sequences=3)
for idx, poem in enumerate(poems, 1):
    print(f"Poem {idx}:\n{poem}\n")


Poem 1:
[قصيدة حزينة] يا أيها الحبيب ما الذي فيك لي
 ويا حبذا اليوم من لا يعلمني
 أن الحب ليس في عينيك بل قد كان كذاك
 أنت لم تطب أبدا على هذا المدى؟! 

 وإن كنت غير بعيد فأنت لست معي؛ فإني إليك كما أنا: إنني سوف أقتلني!!
 إن أحبك

Poem 2:
[قصيدة حزينة] يا ليتني لم أزل على
 منك إلا لياليه فصرت أرى حالي معيدا 
 لا شيء يدوم إلى غير ذاك السرىء الذي يذرعه الوله؟!! ألا ترى أن ذا الدنيا قد خلته من يد النوى ؟ **: سل عن ابن يحيى فأدعو له ب

Poem 3:
[قصيدة حزينة] إذا كانت عيني على غير شجو
 فإن سقتني في صروف الزمان إلى السقام
 فيا ليت شعري هل ما أنال من خمار؟! إن الليل مسهد بأشجانه أو غمامته ؟ 
 أم ماذا يكون لو أسهر الناس عن جفاء ونوحسهم!
 وكم لي



### ✅ *Human Evaluation – Experiment 3*

* *Poem 1:* Very expressive. Smooth rhythm and relevant emotion.
* *Poem 2:* A bit abstract but poetic in flow. Slight logical gaps.
* *Poem 3:* Deep imagery and emotional strength. Clear structure.

*Average Scores:*

* Fluency: 4
* Coherence: 3.5
* Structure: 4
* Emotion: 3.5

*Human Score:* 3.8 / 5


## 3. 7. Quantitative Evaluation

We compute:
- **Loss** on the evaluation set
- **Perplexity** as `exp(loss)` to assess how confidently the model predicts the next token


In [ ]:
import math
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import default_data_collator

# Make sure model is in evaluation mode
model.eval()

# Create a DataLoader for evaluation
eval_loader = DataLoader(
    small_eval_dataset,
    batch_size=4,
    collate_fn=default_data_collator
)

# Track total loss
total_loss = 0
n_batches = 0

# Disable gradient calculation for evaluation
with torch.no_grad():
    for batch in tqdm(eval_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
        loss = outputs.loss

        total_loss += loss.item()
        n_batches += 1

Evaluating: 100%|██████████| 1250/1250 [02:12<00:00,  9.46it/s]


In [ ]:
# Calculate Average Loss
avg_loss = total_loss / n_batches

print(f"\n Evaluation Loss: {avg_loss:.4f}")



 Evaluation Loss: 10.3921


In [ ]:
# Calculate Perplexity
perplexity = math.exp(avg_loss)

print(f" Perplexity: {perplexity:.2f}")

 Perplexity: 32599.92


# 🔴Experiment 4: Full Sample, More Epochs

**Training Configuration:**
- Sample size: **50,000**
- Epochs: **3**
- Learning rate: **5e-5**


## 4.1. Training Setup


## 4.1.1. Load Pretrained AraGPT2

We load the `aubmindlab/aragpt2-base` model and fine-tune it on our poetry data.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("aubmindlab/aragpt2-base")


loading configuration file https://huggingface.co/aubmindlab/aragpt2-base/resolve/main/config.json from cache at /root/.cache/huggingface/transformers/c8489d06657d6f7bbc7c50a9ebb22c5a0d744f1e3abd4c1fb4552c1de84a7d16.c27f94d1fe6a71d31824df935e650dfdda7292d7e26004c12d272f15abbb677c
Model config GPT2Config {
  "_name_or_path": "aubmindlab/aragpt2-base",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 0,
  "embd_pdrop": 0.1,
  "eos_token_id": 0,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls

## 4.1.2. Dataset Splitting and Sampling

We split the dataset into training (90%) and evaluation (10%).  .

In [ ]:
# Split the dataset normally (to respect random selection)
split_dataset = tokenized_datasets["train"].train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# Select a smaller subset for faster training
small_train_dataset = train_dataset.select(range(50000))  # 50,000 examples for training
small_eval_dataset = eval_dataset.select(range(5000))    # 5000 examples for evaluation

print(f"Small train examples: {len(small_train_dataset)}")
print(f"Small eval examples: {len(small_eval_dataset)}")

Small train examples: 50000
Small eval examples: 5000


## 4. 2. Define Training Configuration

We specify:
- Learning rate
- Epochs
- Batch size
- Logging and saving strategy


In [ ]:
from transformers import TrainingArguments, get_scheduler

training_args = TrainingArguments(
    output_dir="./aragpt2-ashaar-finetuned-gpu",
    evaluation_strategy="epoch",
    learning_rate=5e-5,                               # 5e-5 for learning rate
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=3,                               # 3 for epochs
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs",
    report_to="none",
    fp16=True,                                        # Mixed precision (faster on GPU)
    gradient_accumulation_steps=2 ,
    warmup_steps=500,                                 # Gradually increase learning rate
    lr_scheduler_type="linear"
)

PyTorch: setting up devices


## 4. 3. Initialize Trainer

Using Hugging Face’s `Trainer` class to manage training and evaluation.


In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


Using amp half precision backend
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:474: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


## 4. 4. Training the Model

In [ ]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 50000
  Num Epochs = 3
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 2
  Total optimization steps = 9375
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:1949: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx_manager = autocast(dtype=self.amp_dtype)


Epoch,Training Loss,Validation Loss
1,4.330600,4.063473
2,4.180400,3.938462
3,4.046900,3.900045


***** Running Evaluation *****
  Num examples = 5000
  Batch size = 4
Saving model checkpoint to ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125
Configuration saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/config.json
Model weights saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/pytorch_model.bin
tokenizer config file saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/tokenizer_config.json
Special tokens file saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/special_tokens_map.json
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:1949: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx_manager = autocast(dtype=self.amp_dtype)
***** Running Evaluation *****
  Num examples = 5000
  Batch size = 4
Saving model checkpoint to ./aragpt2-ashaar-finetuned-gpu/checkpoint-6250
Configuration saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-6250/config.json
Model weights saved 

TrainOutput(global_step=9375, training_loss=4.3744117578125, metrics={'train_runtime': 5086.1838, 'train_samples_per_second': 29.492, 'train_steps_per_second': 1.843, 'total_flos': 1.95969024e+16, 'train_loss': 4.3744117578125, 'epoch': 3.0})

## 4. 5. Save the Fine-Tuned Model

We save the model and tokenizer for each experiment under distinct output directories (e.g., `experiment1-aragpt2-finetuned`).


In [ ]:
trainer.save_model("./experiment4-aragpt2-finetuned")            # Saves the model
tokenizer.save_pretrained("./experiment4-aragpt2-finetuned")     # Saves the tokenizer


Saving model checkpoint to ./experiment4-aragpt2-finetuned
Configuration saved in ./experiment4-aragpt2-finetuned/config.json
Model weights saved in ./experiment4-aragpt2-finetuned/pytorch_model.bin
tokenizer config file saved in ./experiment4-aragpt2-finetuned/tokenizer_config.json
Special tokens file saved in ./experiment4-aragpt2-finetuned/special_tokens_map.json
tokenizer config file saved in ./experiment4-aragpt2-finetuned/tokenizer_config.json
Special tokens file saved in ./experiment4-aragpt2-finetuned/special_tokens_map.json


('./experiment4-aragpt2-finetuned/tokenizer_config.json',
 './experiment4-aragpt2-finetuned/special_tokens_map.json',
 './experiment4-aragpt2-finetuned/vocab.json',
 './experiment4-aragpt2-finetuned/merges.txt',
 './experiment4-aragpt2-finetuned/added_tokens.json')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./experiment4-aragpt2-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(model_path)

# Move model to GPU manually if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


Didn't find file ./experiment4-aragpt2-finetuned/added_tokens.json. We won't load it.
loading file ./experiment4-aragpt2-finetuned/vocab.json
loading file ./experiment4-aragpt2-finetuned/merges.txt
loading file None
loading file ./experiment4-aragpt2-finetuned/special_tokens_map.json
loading file ./experiment4-aragpt2-finetuned/tokenizer_config.json
loading configuration file ./experiment4-aragpt2-finetuned/config.json
Model config GPT2Config {
  "_name_or_path": "./experiment4-aragpt2-finetuned",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 0,
  "embd_pdrop": 0.1,
  "eos_token_id": 0,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false

## 4. 6. Poetry Generation

We generate multiple Arabic poems given a theme-based prompt like `[قصيدة حزينة]`, using top-k and top-p sampling.


In [ ]:
def generate_multiple_poems(prompt, max_length=70, top_k=40, top_p=0.85, temperature=1.0, num_return_sequences=3):
    device = model.device
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    outputs = model.generate(
        input_ids,
        max_length=max_length,
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.8,
        no_repeat_ngram_size=4,
        early_stopping=True
    )

    poems = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return poems
prompt = "[قصيدة حزينة]"
poems = generate_multiple_poems(prompt, top_k=40, top_p=0.85, num_return_sequences=3)
for idx, poem in enumerate(poems, 1):
    print(f"Poem {idx}:\n{poem}\n")


Poem 1:
[قصيدة حزينة] أيها الناس ما هذا الذي في الكون
 قد صار لنا من غير بابك أو بباب بيتك ؟! يا ذا الشتيتان و أنت طمع؟ كم شبت النار بالنار أمسى لهيبا!! هل هو على الأرض جنة لك أن تغبها؛ لقد ضجت نارهما بالشتا 
 كل يوم فيك

Poem 2:
[قصيدة حزينة] ما لي أن أبث إليك
 من قبل ذا ؟!؟… ماذا أقول! إن كان في قلبي مثل هذا المساء, أم هناك ضوء نهار مظلم:” هل أنت إلا وردة خضراء 
 تلك الوردة التي أغدو معك على مدار السماء, يا قلب الحبيب الذي لم يزل فيه يغمره الظل الغريب … أنا لا

Poem 3:
[قصيدة حزينة] يا من له في الناس يوما
 ما عاد يعاتبني الزمان إلا 
 حتى غدا ذا عقاب!!؟ لقد كنت عندي مثل هذا العقاب ؟ **~2) وما أنت أدرى بما أنا الآن أحسدهم (3 ) ـ بعد أن أقتلناهم بالعقاب, فقد نلقى العذاب عليهم _6



### ✅ *Human Evaluation – Experiment 4*

* *Poem 1:* Complicated expressions. Difficult to follow.
* *Poem 2:* Emotionally expressive but lacks rhythm and clarity.
* *Poem 3:* Thematically scattered with minimal structure.

*Average Scores:*

* Fluency: 2.5
* Coherence: 2
* Structure: 2
* Emotion: 3

*Human Score:* 2.4 / 5


## 4. 7. Quantitative Evaluation

We compute:
- **Loss** on the evaluation set
- **Perplexity** as `exp(loss)` to assess how confidently the model predicts the next token


In [ ]:
import math
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import default_data_collator

# Make sure model is in evaluation mode
model.eval()

# Create a DataLoader for evaluation
eval_loader = DataLoader(
    small_eval_dataset,
    batch_size=4,
    collate_fn=default_data_collator
)

# Track total loss
total_loss = 0
n_batches = 0

# Disable gradient calculation for evaluation
with torch.no_grad():
    for batch in tqdm(eval_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
        loss = outputs.loss

        total_loss += loss.item()
        n_batches += 1

Evaluating: 100%|██████████| 1250/1250 [02:11<00:00,  9.52it/s]


In [ ]:
# Calculate Average Loss
avg_loss = total_loss / n_batches

print(f"\n Evaluation Loss: {avg_loss:.4f}")



 Evaluation Loss: 11.0192


In [ ]:
# Calculate Perplexity
perplexity = math.exp(avg_loss)

print(f" Perplexity: {perplexity:.2f}")

 Perplexity: 61033.32


# 🔴Experiment 5: Full Sample, Higher Learning Rate

**Training Configuration:**
- Sample size: **50,000**
- Epochs: **2**
- Learning rate: **1e-4**



## 5.1. Training Setup


## 5.1.1. Load Pretrained AraGPT2

We load the `aubmindlab/aragpt2-base` model and fine-tune it on our poetry data.

In [7]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("aubmindlab/aragpt2-base")


Downloading:   0%|          | 0.00/527M [00:00<?, ?B/s]

## 5.1.2. Dataset Splitting and Sampling

We split the dataset into training (90%) and evaluation (10%).  .

In [8]:
# Split the dataset normally (to respect random selection)
split_dataset = tokenized_datasets["train"].train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# Select a smaller subset for faster training
small_train_dataset = train_dataset.select(range(50000))  # 50,000 examples for training
small_eval_dataset = eval_dataset.select(range(5000))    # 5000 examples for evaluation

print(f"Small train examples: {len(small_train_dataset)}")
print(f"Small eval examples: {len(small_eval_dataset)}")

Small train examples: 50000
Small eval examples: 5000


## 5. 2. Define Training Configuration

We specify:
- Learning rate
- Epochs
- Batch size
- Logging and saving strategy


In [9]:
from transformers import TrainingArguments, get_scheduler

training_args = TrainingArguments(
    output_dir="./aragpt2-ashaar-finetuned-gpu",
    evaluation_strategy="epoch",
    learning_rate=1e-4,                             # 1e-4 for learning rate
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=2,                             # 2 for epochs
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs",
    report_to="none",
    fp16=True,                                      # Mixed precision (faster on GPU)
    gradient_accumulation_steps=2 ,
    warmup_steps=500,                               # Gradually increase learning rate
    lr_scheduler_type="linear"
)

## 5. 3. Initialize Trainer

Using Hugging Face’s `Trainer` class to manage training and evaluation.


In [10]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


Using amp half precision backend
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:474: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


## 5. 4. Training the Model

In [11]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 50000
  Num Epochs = 2
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 2
  Total optimization steps = 6250
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:1949: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx_manager = autocast(dtype=self.amp_dtype)


Epoch,Training Loss,Validation Loss
1,4.209100,3.985009
2,4.047500,3.867959


***** Running Evaluation *****
  Num examples = 5000
  Batch size = 4
Saving model checkpoint to ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125
Configuration saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/config.json
Model weights saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/pytorch_model.bin
tokenizer config file saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/tokenizer_config.json
Special tokens file saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/special_tokens_map.json
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:1949: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx_manager = autocast(dtype=self.amp_dtype)
***** Running Evaluation *****
  Num examples = 5000
  Batch size = 4
Saving model checkpoint to ./aragpt2-ashaar-finetuned-gpu/checkpoint-6250
Configuration saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-6250/config.json
Model weights saved 

TrainOutput(global_step=6250, training_loss=4.36777119140625, metrics={'train_runtime': 3242.7758, 'train_samples_per_second': 30.838, 'train_steps_per_second': 1.927, 'total_flos': 1.30646016e+16, 'train_loss': 4.36777119140625, 'epoch': 2.0})

## 5. 5. Save the Fine-Tuned Model

We save the model and tokenizer for each experiment under distinct output directories (e.g., `experiment1-aragpt2-finetuned`).


In [12]:
trainer.save_model("./experiment5-aragpt2-finetuned")           # Saves the model
tokenizer.save_pretrained("./experiment5-aragpt2-finetuned")    # Saves the tokenizer


Saving model checkpoint to ./experiment5-aragpt2-finetuned
Configuration saved in ./experiment5-aragpt2-finetuned/config.json
Model weights saved in ./experiment5-aragpt2-finetuned/pytorch_model.bin
tokenizer config file saved in ./experiment5-aragpt2-finetuned/tokenizer_config.json
Special tokens file saved in ./experiment5-aragpt2-finetuned/special_tokens_map.json
tokenizer config file saved in ./experiment5-aragpt2-finetuned/tokenizer_config.json
Special tokens file saved in ./experiment5-aragpt2-finetuned/special_tokens_map.json


('./experiment5-aragpt2-finetuned/tokenizer_config.json',
 './experiment5-aragpt2-finetuned/special_tokens_map.json',
 './experiment5-aragpt2-finetuned/vocab.json',
 './experiment5-aragpt2-finetuned/merges.txt',
 './experiment5-aragpt2-finetuned/added_tokens.json')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./experiment5-aragpt2-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(model_path)

# Move model to GPU manually if available:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


## 5. 6. Poetry Generation

We generate multiple Arabic poems given a theme-based prompt like `[قصيدة حزينة]`, using top-k and top-p sampling.


In [14]:
def generate_multiple_poems(prompt, max_length=70, top_k=40, top_p=0.85, temperature=1.0, num_return_sequences=3):
    device = model.device
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    outputs = model.generate(
        input_ids,
        max_length=max_length,
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.8,
        no_repeat_ngram_size=4,
        early_stopping=True
    )

    poems = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return poems
prompt = "[قصيدة حزينة]"
poems = generate_multiple_poems(prompt, top_k=40, top_p=0.85, num_return_sequences=3)
for idx, poem in enumerate(poems, 1):
    print(f"Poem {idx}:\n{poem}\n")


Poem 1:
[قصيدة حزينة] يا أيها الملك الذي قد حوت من يديه
 لا زلت تنفض عني ولا تطيعني فإني في الهوى مجنونا أبدا 
 فإن أنا قلت اليوم لم أكن لي شيئا
 إن كنت تدري أن الدهر سوف يأتي غدا عليك قريبا فمنه إليك فلا تخف فإنني سأبصره!؟! هل ترى الآن أني أقضي عمري

Poem 2:
[قصيدة حزينة] يا ليتني لم أزل أبكي
 حتى كأني قد كنت أعبر!؟ من لي بمن كان أعلم ؟ ومن أنا بعهده: ألم به الآن! كيف أنسى وما نسيتم ؛ أين الذي فيكم؛ وكيف تنسى كل مساءاتكم, في أي يوم يأتيها عيد جديد 
 إن العيد أضحى عيدا جديدا وقد جاء

Poem 3:
[قصيدة حزينة] إذا كانت عيني على غير شكاية
 فإن لم يكن لي في الناس خير صديق 
 وإن كنت عن كل يوم ضيف غريب º فالليل سقام طويل„ يهـا نميره¦ ثقيل! قد كان من الدهر مقصر؟ قلت لا والله ما أنا إذ أتيتك
 ولا



### ✅ *Human Evaluation – Experiment 5*

* *Poem 1:* Starts strong, then wanders. Abstract ideas with some logical gaps.
* *Poem 2:* Poetic language with metaphorical depth.
* *Poem 3:* Surprisingly coherent and emotional despite some noise.

*Average Scores:*

* Fluency: 3.5
* Coherence: 3
* Structure: 3.2
* Emotion: 3.2

*Human Score:* 3.2 / 5


## 5. 7. Quantitative Evaluation

We compute:
- **Loss** on the evaluation set
- **Perplexity** as `exp(loss)` to assess how confidently the model predicts the next token


In [15]:
import math
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import default_data_collator

# Make sure model is in evaluation mode
model.eval()

# Create a DataLoader for evaluation
eval_loader = DataLoader(
    small_eval_dataset,
    batch_size=4,
    collate_fn=default_data_collator
)

# Track total loss
total_loss = 0
n_batches = 0

# Disable gradient calculation for evaluation
with torch.no_grad():
    for batch in tqdm(eval_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
        loss = outputs.loss

        total_loss += loss.item()
        n_batches += 1

Evaluating: 100%|██████████| 1250/1250 [02:05<00:00,  9.95it/s]


In [16]:
# Calculate Average Loss
avg_loss = total_loss / n_batches

print(f"\n Evaluation Loss: {avg_loss:.4f}")



 Evaluation Loss: 10.7590


In [17]:
# Calculate Perplexity
perplexity = math.exp(avg_loss)

print(f" Perplexity: {perplexity:.2f}")

 Perplexity: 47049.80


# 🔴Experiment 6: Full Sample, Higher LR, More Epochs

**Training Configuration:**
- Sample size: **50,000**
- Epochs: **3**
- Learning rate: **1e-4**


## 6.1. Training Setup


## 6.1.1. Load Pretrained AraGPT2

We load the `aubmindlab/aragpt2-base` model and fine-tune it on our poetry data.

In [7]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("aubmindlab/aragpt2-base")


Downloading:   0%|          | 0.00/527M [00:00<?, ?B/s]

## 6.1.2. Dataset Splitting and Sampling

We split the dataset into training (90%) and evaluation (10%).  .

In [8]:
# Split the dataset normally (to respect random selection)
split_dataset = tokenized_datasets["train"].train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# Select a smaller subset for faster training
small_train_dataset = train_dataset.select(range(50000))  # 50,000 examples for training
small_eval_dataset = eval_dataset.select(range(5000))    # 5000 examples for evaluation

print(f"Small train examples: {len(small_train_dataset)}")
print(f"Small eval examples: {len(small_eval_dataset)}")

Small train examples: 50000
Small eval examples: 5000


## 6. 2. Define Training Configuration

We specify:
- Learning rate
- Epochs
- Batch size
- Logging and saving strategy


In [9]:
from transformers import TrainingArguments, get_scheduler

training_args = TrainingArguments(
    output_dir="./aragpt2-ashaar-finetuned-gpu",
    evaluation_strategy="epoch",
    learning_rate=1e-4,                             # 5e-5 for learning rate
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=3,                             # 3 for epochs
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs",
    report_to="none",
    fp16=True,                                      # Mixed precision (faster on GPU)
    gradient_accumulation_steps=2 ,
    warmup_steps=500,                               # Gradually increase learning rate
    lr_scheduler_type="linear"
)

## 6. 3. Initialize Trainer

Using Hugging Face’s `Trainer` class to manage training and evaluation.


In [10]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


Using amp half precision backend
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:474: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


## 6. 4. Training the Model

In [11]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 50000
  Num Epochs = 3
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 2
  Total optimization steps = 9375
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:1949: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx_manager = autocast(dtype=self.amp_dtype)


Epoch,Training Loss,Validation Loss
1,4.205400,3.986801
2,4.022100,3.839205
3,3.854900,3.788328


***** Running Evaluation *****
  Num examples = 5000
  Batch size = 4
Saving model checkpoint to ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125
Configuration saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/config.json
Model weights saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/pytorch_model.bin
tokenizer config file saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/tokenizer_config.json
Special tokens file saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-3125/special_tokens_map.json
/usr/local/lib/python3.11/dist-packages/transformers/trainer.py:1949: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx_manager = autocast(dtype=self.amp_dtype)
***** Running Evaluation *****
  Num examples = 5000
  Batch size = 4
Saving model checkpoint to ./aragpt2-ashaar-finetuned-gpu/checkpoint-6250
Configuration saved in ./aragpt2-ashaar-finetuned-gpu/checkpoint-6250/config.json
Model weights saved 

TrainOutput(global_step=9375, training_loss=4.2025202083333335, metrics={'train_runtime': 4856.1624, 'train_samples_per_second': 30.889, 'train_steps_per_second': 1.931, 'total_flos': 1.95969024e+16, 'train_loss': 4.2025202083333335, 'epoch': 3.0})

## 6. 5. Save the Fine-Tuned Model

We save the model and tokenizer for each experiment under distinct output directories (e.g., `experiment1-aragpt2-finetuned`).


In [12]:
trainer.save_model("./experiment6-aragpt2-finetuned")             # Saves the model
tokenizer.save_pretrained("./experiment6-aragpt2-finetuned")      # Saves the tokenizer


Saving model checkpoint to ./experiment6-aragpt2-finetuned
Configuration saved in ./experiment6-aragpt2-finetuned/config.json
Model weights saved in ./experiment6-aragpt2-finetuned/pytorch_model.bin
tokenizer config file saved in ./experiment6-aragpt2-finetuned/tokenizer_config.json
Special tokens file saved in ./experiment6-aragpt2-finetuned/special_tokens_map.json
tokenizer config file saved in ./experiment6-aragpt2-finetuned/tokenizer_config.json
Special tokens file saved in ./experiment6-aragpt2-finetuned/special_tokens_map.json


('./experiment6-aragpt2-finetuned/tokenizer_config.json',
 './experiment6-aragpt2-finetuned/special_tokens_map.json',
 './experiment6-aragpt2-finetuned/vocab.json',
 './experiment6-aragpt2-finetuned/merges.txt',
 './experiment6-aragpt2-finetuned/added_tokens.json')

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./experiment6-aragpt2-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(model_path)

# Move model to GPU manually if available:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


Didn't find file ./experiment6-aragpt2-finetuned/added_tokens.json. We won't load it.
loading file ./experiment6-aragpt2-finetuned/vocab.json
loading file ./experiment6-aragpt2-finetuned/merges.txt
loading file None
loading file ./experiment6-aragpt2-finetuned/special_tokens_map.json
loading file ./experiment6-aragpt2-finetuned/tokenizer_config.json
loading configuration file ./experiment6-aragpt2-finetuned/config.json
Model config GPT2Config {
  "_name_or_path": "./experiment6-aragpt2-finetuned",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 0,
  "embd_pdrop": 0.1,
  "eos_token_id": 0,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false

## 6. 6. Poetry Generation

We generate multiple Arabic poems given a theme-based prompt like `[قصيدة حزينة]`, using top-k and top-p sampling.


In [14]:
def generate_multiple_poems(prompt, max_length=70, top_k=40, top_p=0.85, temperature=1.0, num_return_sequences=3):
    device = model.device
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    outputs = model.generate(
        input_ids,
        max_length=max_length,
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.8,
        no_repeat_ngram_size=4,
        early_stopping=True
    )

    poems = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return poems
prompt = "[قصيدة حزينة]"
poems = generate_multiple_poems(prompt, top_k=40, top_p=0.85, num_return_sequences=3)
for idx, poem in enumerate(poems, 1):
    print(f"Poem {idx}:\n{poem}\n")


Poem 1:
[قصيدة حزينة] أيها الناس لا تعذروا
 إن ما في الأرض من كرم كريم 
 أوفى على المرء وهو خير كبير ؟!؟ نعم أنت الذي كان لنا مثلا قديما
 لم ير مثله هذا الدهر يوما طويلا! هل هو إلا الزمان القديم
 كم شادتنا الدنيا وما فيها دهورا ونوما معا ؟! أم أن

Poem 2:
[قصيدة حزينة] أما وقد صرت في الدهر أسيرا
 من قبل أن أموت حرا طليقا
 ما بال حالي إن كان مستسلما 
 أمسى عليلا متصفا بأساقم؟! ألا يا قوم هل ترى شيئا لي غيركم ؟! لقد صح عندي منكم أني حزينه "يا رباه" ألم يكن ذنبي إلا

Poem 3:
[قصيدة حزينة] يا من له في الناس يوما
 ما أنت إلا بعض موتى بهاته ال 
 ت لا شك عندي سوى ذاك راتلناه! **1)
 إني سأرضى عنك إن كنت حياليتي؟ فعد لي يلذ لنا: أما ترى لك قائمه ؟ "يا أيها المليك الذي



### ✅ *Human Evaluation – Experiment 6*

* *Poem 1:* Hard to interpret. Lacks clarity and emotional engagement.
* *Poem 2:* Confusing transitions. Little coherence.
* *Poem 3:* Excessively verbose with forced phrases.

*Average Scores:*

* Fluency: 2.2
* Coherence: 2
* Structure: 2
* Emotion: 2.5

*Human Score:* 2.2 / 5


## 6. 7. Quantitative Evaluation

We compute:
- **Loss** on the evaluation set
- **Perplexity** as `exp(loss)` to assess how confidently the model predicts the next token


In [15]:
import math
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import default_data_collator

# Make sure model is in evaluation mode
model.eval()

# Create a DataLoader for evaluation
eval_loader = DataLoader(
    small_eval_dataset,
    batch_size=4,
    collate_fn=default_data_collator
)

# Track total loss
total_loss = 0
n_batches = 0

# Disable gradient calculation for evaluation
with torch.no_grad():
    for batch in tqdm(eval_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
        loss = outputs.loss

        total_loss += loss.item()
        n_batches += 1

Evaluating: 100%|██████████| 1250/1250 [02:05<00:00,  9.97it/s]


In [16]:
# Calculate Average Loss
avg_loss = total_loss / n_batches

print(f"\n Evaluation Loss: {avg_loss:.4f}")



 Evaluation Loss: 11.6218


In [17]:
# Calculate Perplexity
perplexity = math.exp(avg_loss)

print(f" Perplexity: {perplexity:.2f}")

 Perplexity: 111505.75




---



### ✅ Summary of Experimental Results

| Experiment | Sample Size | Epochs | Learning Rate | Human Score (/5) | Eval Loss | Perplexity |
|------------|-------------|--------|----------------|------------------|-----------|------------|
| **1**      | 5K          | 2      | 5e-5           | 2.5              | 7.7391    | 2296.43    |
| **2**      | 10K         | 2      | 5e-5           | 3.2              | 8.5515    | 5174.41    |
| **3**      | 50K         | 2      | 5e-5           | **3.8**          | 10.3921   | 32599.92   |
| **4**      | 50K         | 3      | 5e-5           | 2.4              | 11.0192   | 61033.32   |
| **5**      | 50K         | 2      | 1e-4           | 3.2              | 10.7590   | 47049.80   |
| **6**      | 50K         | 3      | 1e-4           | 2.2              | 11.6218   | 111505.75  |



### 🔍 Key Takeaways

- **Experiment 3** achieved the **highest human score (3.8)** despite higher loss and perplexity, showing that **quantitative metrics don’t always align with human perception**.
- Increasing **sample size** improved poem quality more than tweaking **epochs** or **learning rate**.
- **Higher epochs and learning rate** in Experiment 6 led to degraded output, suggesting **overfitting or unstable learning**.
- **Human evaluation** is essential for subjective tasks like poetry, where emotional impact matters more than perplexity.
